In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os

base_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
train_dir = os.path.join(base_dir, "data", "asl_alphabet_train", "asl_alphabet_train")
if not os.path.exists(train_dir):
    train_dir = "data/asl_alphabet_train/asl_alphabet_train"

img_size = (64, 64)      # smaller = faster on CPU
batch_size = 32

exclude = {"space", "del", "nothing"}
all_classes = sorted([c for c in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, c))])
alphabet_classes = [c for c in all_classes if c not in exclude]

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.15, subset="training",
    seed=123, image_size=img_size, batch_size=batch_size,
    class_names=alphabet_classes
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.15, subset="validation",
    seed=123, image_size=img_size, batch_size=batch_size,
    class_names=alphabet_classes
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Training on {num_classes} classes: {class_names}")

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
])

normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Found 78000 files belonging to 26 classes.
Using 66300 files for training.
Found 78000 files belonging to 26 classes.
Using 11700 files for validation.
Training on 26 classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [2]:
import tensorflow as tf
print("GPUs available:", tf.config.list_physical_devices('GPU'))

GPUs available: []


In [3]:
epochs = 30

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras', monitor='val_accuracy', save_best_only=True
    )
]

model = models.Sequential([
    layers.Input(shape=(64, 64, 3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=callbacks
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 26)             │         6,682 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,279,834 (4.88 MB)

 Trainable params: 1,279,834 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
2072/2072 ━━━━━━━━━━━━━━━━━━━━ 153s 64ms/step - accuracy: 0.5883 - loss: 1.3112 - val_accuracy: 0.8484 - val_loss: 0.4200
Epoch 2/30
2072/2072 ━━━━━━━━━━━━━━━━━━━━ 150s 60ms/step - accuracy: 0.8578 - loss: 0.4127 - val_accuracy: 0.9485 - val_loss: 0.1573
Epoch 3/30
2072/2072 ━━━━━━━━━━━━━━━━━━━━ 145s 60ms/step - accuracy: 0.9068 - loss: 0.2678 - val_accuracy: 0.9629 - val_loss: 0.1159
Epoch 4/30
2072/2072 ━━━━━━━━━━━━━━━━━━━━ 150s 63ms/step - accuracy: 0.9271 - loss: 0.2099 - val_accuracy: 0.9820 - val_loss: 0.0541
Epoch 5/30
2072/2072 ━━━━━━━━━━━━━━━━━━━━ 151s 62ms/step - accuracy: 0.9395 - loss: 0.1762 - val_accuracy: 0.9738 - val_loss: 0.0702
Epoch 6/30
2072/2072 ━━━━━━━━━━━━━━━━━━━━ 145s 60ms/step - accuracy: 0.9480 - loss: 0.1517 - val_accuracy: 0.9864 - val_loss: 0.0402
Epoch 7/30
2072/2072 ━━━━━━━━━━━━━━━━━━━━ 141s 59ms/step - accuracy: 0.9535 - loss: 0.1370 - val_accuracy: 0.9838 - val_loss: 0.0453
Epoch 8/30
2072/2072 ━━━━━━━━━━━━━━━━━━━━ 161s 68ms/step - accuracy: 

In [4]:
import os
import datetime

mtime = os.path.getmtime("best_model.keras")
print("Last modified:", datetime.datetime.fromtimestamp(mtime))

Last modified: 2026-08-30 18:26:13.438032
